# Benchmark of LLM agent performance in uncertain environments
This notebook analyzes the performance of an llm trying to navigate an uncertain environment. Using a lookahead search it decides which steps are the best possible moves. To value each step another llm is leveraged with a list of hypothese and policies about the environment.


In [1]:
# Setup + Imports
from evaluation.Benchmark import Benchmark
from openai import OpenAI
import gymnasium as gym
from tinydb import TinyDB
import os
from navigation.environments.FrozenLakeEnv import FrozenLakeEnv
from navigation.environments.FrozenLakeShadowEnv import FrozenLakeShadowEnv
from optimization.prompts.FrozenLakePrompts import FrozenLakePrompts

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
)
model = "openai/gpt-oss-120b"

## 1. FrozenLake Benchmarks

In [ ]:
map_4x4 = ["SFFF","FFHF","HFFF","HHFG"]
map_5x5 = ["SFFFH","FFFHF","FHFFF","FFFFF","FFFFG"]
map_6x6 = ["SFFFFH", "FFFFFF", "FHFFFF", "HFFFFH", "FHFFFF", "FFFFFG"]
map_7x7 = ["SFHFFFF","FFFHFFH","FFFFFFF","FFFHFFF","FFFHFFF","FFFFFFF","HFFFFFG"]
map_8x8 = ["SFFFFFFH", "FFFFHFFF", "FFFFFFFF", "FFFHFFFH", "FFFFHFFH", "FFHFFFHF", "FFFFFFFF", "FFFFFFFG"]
map_list = [map_4x4, map_5x5, map_6x6, map_7x7, map_8x8]

policyDb = TinyDB("../store/policies.json")
hypothesesDb = TinyDB("../store/hypotheses.json")

sucess_rate=0.7
reward_schedule=(1, 0, 0)
num_episodes = 10

debug = True
map_list = [map_8x8]

for map in map_list:
    policyDb.truncate()
    hypothesesDb.truncate()
    #setup
    map_size = len(map)
    map_name = f"{map_size}x{map_size}"

    env = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=map, is_slippery=False, success_rate=sucess_rate, reward_schedule=reward_schedule))
    env_slippery = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=map, is_slippery=True, success_rate=sucess_rate, reward_schedule=reward_schedule))
    shadow_env = FrozenLakeShadowEnv(hypothesesDb=hypothesesDb, policyDb=policyDb, client=client, model=model)
    optimizationPrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

    bench = Benchmark(env=env, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
    bench_slippery = Benchmark(env=env_slippery, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
    benchmarks = {f"{map_name}":bench, f"{map_name}_slippery={sucess_rate}":bench_slippery}

    MAX_STEPS = map_size ** 2

    print(f"# Benchmarks for {map_name} map with success_rate={sucess_rate} and reward_schedule={reward_schedule} #")
    # look=false, shadow=copy
    #for name, bench in benchmarks.items():
    #    print(f"## Executing {name} with look=false, shadow=copy:")
    #    policyDb.truncate()
    #    hypothesesDb.truncate()
    #    bench.run(name=name, iteration_depth=num_episodes, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=debug)

    # look=true, shadow=copy: 
    #for name, bench in benchmarks.items():
    #    print(f"## Executing {name} with look=true, shadow=copy:")
    #    policyDb.truncate()
    #    hypothesesDb.truncate()
    #    bench.run(name=name, iteration_depth=num_episodes, lookahead_sample_size=2, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=debug)

    # look=false, shadow=llm
    #for name, bench in benchmarks.items():
    #    print(f"## Executing {name} with look=false, shadow=llm:")
    #    policyDb.truncate()
    #    hypothesesDb.truncate()
    #    bench.run(name=name, iteration_depth=num_episodes, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=debug)

    # look=true, shadow=llm:TODO: for 8x8 run the slippery version and everything after that
    for name, bench in benchmarks.items():
        print(f"## Executing {name} with look=true, shadow=llm:")
        policyDb.truncate()
        hypothesesDb.truncate()
        bench.run(name=name, iteration_depth=num_episodes, lookahead_sample_size=2, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=debug)

# Benchmarks for 8x8 map with success_rate=0.7 and reward_schedule=(1, 0, 0) #
## Executing 8x8 with look=false, shadow=llm:
Agent Response: ChatCompletionMessage(content='{"move": "move_down", "value": 85}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to evaluate moves. Grid 8x8? Let\'s map coordinates. Row0: "[S] F  F  F  F  F  F  H". So start at (0,0). Goal G at last row (7) column 7. Hazards H positions need avoidance. We consider move_right to (0,1) safe (F). move_down to (1,0) safe (F). move_left/up would go out of bounds, likely invalid? Not specified but likely not allowed (would be outside grid). Could be considered moving away or invalid. So we need best move toward goal while avoiding hazards. Hazards at many positions: coordinates (0,7), (1,4), (3,3), (3,7), (4,4), (4,7), (5,2), (5,6), (6,... none), (7,... none). Starting from (0,0) moving right or down both move toward goal. Both safe. Which is bette

### 1.1 Small Map 4x4

#### 1.1.1 Setup

In [ ]:
policyDb = TinyDB("../store/policies.json")
hypothesesDb = TinyDB("../store/hypotheses.json")

map_4x4 = [
"SFFF",
"FFHF",
"HFFF",
"HHFG"
]

env = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=map_4x4, is_slippery=False, success_rate=0.7, reward_schedule=(1, 0, 0)))
env_slippery = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=map_4x4, is_slippery=True, success_rate=0.7, reward_schedule=(1, 0, 0)))
shadow_env = FrozenLakeShadowEnv(hypothesesDb=hypothesesDb, policyDb=policyDb, client=client, model=model)
optimizationPrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

bench_smallMap = Benchmark(env=env, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
bench_smallMap_slippery = Benchmark(env=env_slippery, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
policyDb.truncate()
hypothesesDb.truncate()

MAX_STEPS = 30

#### 1.1.2 Lookahead=False, ShadowEnv=copy(), Slippery=False

In [ ]:
metrics_nlook_nllm_small = bench_smallMap.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.3 Lookahead=False, ShadowEnv=copy(), Slippery=True

In [ ]:
metrics_nlook_nllm_small = bench_smallMap_slippery.run(name="frozenlake_slippery=0.7_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.4 Lookahead=False, ShadowEnv=llm, Slippery=False 

In [ ]:
metrics_nlook_llm_small = bench_smallMap.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.5 Lookahead=False, ShadowEnv=llm, Slippery=True

In [ ]:
metrics_nlook_llm_small = bench_smallMap_slippery.run(name="frozenlake_slippery=0.7_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.6 Lookahead=True, ShadowEnv=copy(), Slippery=False

In [ ]:
metrics_2look_nllm_small = bench_smallMap.run(name="frozenlake7_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.7 Lookahead=True, ShadowEnv=copy(), Slippery=True

In [ ]:
metrics_2look_nllm_small = bench_smallMap_slippery.run(name="frozenlake_slippery=0.7_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.8 Lookahead=True, ShadowEnv=llm, Slippery=False

In [ ]:
metrics_2look_llm_small = bench_smallMap.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.1.9 Lookahead=True, ShadowEnv=llm, Slippery=True

In [ ]:

metrics_2look_llm_small = bench_smallMap_slippery.run(name="frozenlake_slippery=0.7_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

### 1.2 Big Map 8x8

#### 1.2.1 Setup

In [ ]:
policyDb = TinyDB("../store/policies.json")
hypothesesDb = TinyDB("../store/hypotheses.json")

env = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=None, map_name="8x8", is_slippery=False, success_rate=0.7, reward_schedule=(1, 0, 0)))
env_slippery = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=None, map_name="8x8", is_slippery=True, success_rate=0.7, reward_schedule=(1, 0, 0)))
shadow_env = FrozenLakeShadowEnv(hypothesesDb=hypothesesDb, policyDb=policyDb, client=client, model=model)
optimizationPrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

bench_bigMap_noLlm = Benchmark(env=env, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
bench_bigMap_noLlm_slippery = Benchmark(env=env_slippery, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
policyDb.truncate()
hypothesesDb.truncate()

MAX_STEPS = 60

#### 1.2.2 Lookahead=False, ShadowEnv=copy(), Slippery=False

In [ ]:
metrics_nlook_nllm_big = bench_bigMap_noLlm.run(name="frozenlake_big", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.2.3 Lookahead=False, ShadowEnv=copy(), Slippery=True

In [ ]:
metrics_nlook_nllm_big_slippery = bench_bigMap_noLlm_slippery.run(name="frozenlake_slippery=0.7_big", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.2.4 Lookahead=True, ShadowEnv=copy(), Slippery=False

In [ ]:
metrics_2look_nllm_big = bench_bigMap_noLlm.run(name="frozenlake7_big", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.2.5 Lookahead=True, ShadowEnv=copy(), Slippery=True

In [ ]:

metrics_2look_nllm_big_slippery = bench_bigMap_noLlm_slippery.run(name="frozenlake_slippery=0.7_big", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=False, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.2.6 Lookahead=False, ShadowEnv=llm, Slippery=False

In [ ]:
# no lookahead, llm action, big map
metrics_nlook_llm_big = bench_bigMap_noLlm.run(name="frozenlake_big", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()


#### 1.2.7 Lookahead=False, ShadowEnv=llm, Slippery=True

In [ ]:

metrics_nlook_llm_big_slippery = bench_bigMap_noLlm_slippery.run(name="frozenlake_slippery=0.7_big", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.2.8 Lookahead=True, ShadowEnv=llm, Slippery=False

In [ ]:
metrics_2look_llm_big = bench_bigMap_noLlm.run(name="frozenlake7_big", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()

#### 1.2.9 Lookahead=True, ShadowEnv=llm, Slippery=True

In [ ]:
metrics_2look_llm_big_slippery = bench_bigMap_noLlm_slippery.run(name="frozenlake_slippery=0.7_big", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=MAX_STEPS, use_llm_action=True, print_debug=False)
policyDb.truncate()
hypothesesDb.truncate()